# Project 1: Malaria Cell Detection on Edge Devices
Classify blood cell images as *Parasitized* or *Uninfected* with a small MobileNetV2, convert it to TensorFlow Lite, and measure accuracy, sensitivity, specificity, size and CPU speed.
Runtime: *Runtime > Change runtime type > T4 GPU*.


## Step 1: Setup
Mount Drive, import libraries, create the save folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import time
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix

save_dir = "/content/drive/MyDrive/Malaria_Edge_Project"
os.makedirs(save_dir, exist_ok=True)

## Step 2: Load the dataset
NIH malaria dataset (27,558 images), 80% train / 20% validation. Check which label number is which class.

In [ ]:
(raw_train, raw_val), ds_info = tfds.load(
    'malaria',
    split=['train[:80%]', 'train[80%:]'],
    as_supervised=True,
    with_info=True
)
print("Classes:", ds_info.features['label'].names)  # index 0 and 1

## Step 3: Input pipeline
Resize to 96x96, cache, shuffle (train only), batch, prefetch.

In [ ]:
IMG_SIZE = (96, 96)
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    return image, label

train_ds = (raw_train.map(preprocess, num_parallel_calls=AUTOTUNE)
            .cache().shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE))
val_ds = (raw_val.map(preprocess, num_parallel_calls=AUTOTUNE)
          .cache().batch(BATCH_SIZE).prefetch(AUTOTUNE))

## Step 4: Model
Frozen MobileNetV2 (pretrained on ImageNet) + small classifier head. `Rescaling(1/127.5, offset=-1)` maps pixels from [0, 255] to [-1, 1], which MobileNetV2 expects.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96, 96, 3), include_top=False, weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(96, 96, 3)),
    layers.Rescaling(1./127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

## Step 5: Train

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=5)

## Step 6: Clinical metrics
Parasitized (label 0) is treated as the positive case.
- Sensitivity = TP / (TP + FN)
- Specificity = TN / (TN + FP)

In [ ]:
y_true, y_pred_probs = [], []
for images, labels in val_ds:
    y_true.extend(labels.numpy())
    y_pred_probs.extend(model.predict(images, verbose=0).flatten())

y_true = np.array(y_true)
y_pred = (np.array(y_pred_probs) > 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tp, fn, fp, tn = cm.ravel()   # class 0 (Parasitized) = positive

sensitivity = tp / (tp + fn) * 100
specificity = tn / (tn + fp) * 100
accuracy = (tp + tn) / len(y_true) * 100
print(cm)
print(f"Accuracy {accuracy:.2f}% | Sensitivity {sensitivity:.2f}% | Specificity {specificity:.2f}%")

## Step 7: Convert to TensorFlow Lite (quantized)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = os.path.join(save_dir, "malaria_edge_quantized.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

model_size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f"Model size: {model_size_mb:.2f} MB")

## Step 8: CPU latency benchmark
Warm up first, then average 100 timed runs. This uses a random image; for a stricter test, time real validation images.

In [ ]:
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()

dummy_input = np.random.random_sample((1, 96, 96, 3)).astype(np.float32)
for _ in range(10):
    interpreter.set_tensor(input_details[0]['index'], dummy_input)
    interpreter.invoke()

latencies = []
for _ in range(100):
    start = time.perf_counter()
    interpreter.set_tensor(input_details[0]['index'], dummy_input)
    interpreter.invoke()
    latencies.append((time.perf_counter() - start) * 1000)

avg_latency_ms = np.mean(latencies)
print(f"Average latency: {avg_latency_ms:.2f} ms")

## Step 9: Save the results summary

In [ ]:
summary_text = (
    "=============================================\n"
    "MALARIA EDGE MODEL RESULTS\n"
    "=============================================\n"
    f"Total Validation Samples : {len(y_true)}\n"
    f"Overall Accuracy         : {accuracy:.2f}%\n"
    f"Sensitivity (Recall)     : {sensitivity:.2f}%\n"
    f"Specificity              : {specificity:.2f}%\n"
    f"Quantized Model Size     : {model_size_mb:.2f} MB\n"
    f"Avg CPU Inference Time   : {avg_latency_ms:.2f} ms/cell\n"
    "=============================================\n"
)
print(summary_text)

with open(os.path.join(save_dir, "abstract_metrics.txt"), "w") as f:
    f.write(summary_text)
print(f"Saved to: {save_dir}")

## Limits to state honestly
One dataset, one random split, no external test set, and the same images from one patient can appear in both train and validation.